In [10]:
import os
import numpy as np
import networkx as nx
from rdkit import Chem
from gspan import GSpan

In [11]:
print('Loading NCI dataset')
DATASET_DIR = "datasets/NCI_full"  # change this
graphs = []
y = []

filepath = "datasets/NCI_full/1total-connect.sdf"

supplier = Chem.SDMolSupplier(filepath, sanitize=False, removeHs=True)
for mol in supplier:
    if mol is None:
        continue

    G = nx.Graph()

    # Add atoms as nodes
    for atom in mol.GetAtoms():
        G.add_node(
            atom.GetIdx(),
            feature=atom.GetSymbol()   # WL uses node labels
        )

    # Add bonds as edges
    for bond in mol.GetBonds():
        G.add_edge(
            bond.GetBeginAtomIdx(),
            bond.GetEndAtomIdx(),
            bond_type=str(bond.GetBondType()),
            bond_order=bond.GetBondTypeAsDouble(),
            aromatic=bond.GetIsAromatic(),
            in_ring=bond.IsInRing(),
            conjugated=bond.GetIsConjugated(),
            stereo=str(bond.GetStereo())
        )

    # Get graph label
    # In NCI, class label is stored as a molecule property
    label = int(float(mol.GetProp("value")))
    graphs.append(G)
    y.append(label)

print(f"Loaded {len(graphs)} graphs")

Loading NCI dataset


[19:08:24] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.
[19:08:26] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.
[19:08:28] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.
[19:08:29] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.
[19:08:31] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.
[19:08:31] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.


Loaded 37349 graphs


In [12]:
# ===================================================================
# 2. Inspect edge label distribution (do this first)
# ===================================================================
from collections import Counter

bond_type_counts = Counter()
for G in graphs:
    for u, v, data in G.edges(data=True):
        bond_type_counts[data['bond_type']] += 1

print("Bond type distribution:")
for bt, count in bond_type_counts.most_common():
    print(f"{bt}: {count}")
print(f"Unique bond types: {len(bond_type_counts)}")

Bond type distribution:
SINGLE: 766387
DOUBLE: 291974
TRIPLE: 3958
Unique bond types: 3


In [13]:
sizes = [G.number_of_nodes() for G in graphs]
print(f"Min: {min(sizes)}, Max: {max(sizes)}, "
      f"Mean: {np.mean(sizes):.1f}, Median: {np.median(sizes):.1f}")

Min: 3, Max: 229, Mean: 26.2, Median: 24.0


In [14]:
# Set support as a percentage of your actual dataset
min_support_ratio = 0.05  # 5% — tune this
min_support = int(len(graphs) * min_support_ratio)

print(f"Dataset size: {len(graphs)}")
print(f"min_support: {min_support} ({min_support_ratio*100:.0f}%)")

Dataset size: 37349
min_support: 1867 (5%)


In [15]:
# ===================================================================
# 3. Run gSpan
# ===================================================================

miner = GSpan(
    min_support=min_support,       # ~5% of dataset; tune as needed
    max_num_vertices=12,    # limit pattern size for tractability (the maximum number of vertices (atoms) allowed in any mined subgraph pattern)
    verbose=True
)
# Patterns larger than your smallest graph can never have full support, so that sets a loose upper bound.
miner.run(
    graphs,
    node_label_attr='feature',
    edge_label_attr='bond_type',  # single attribute -> clean label space
)

[gSpan] Database: 37349 graphs, 64 vertex labels, 3 edge labels
[gSpan] Frequent 1-edge subgraphs: 16
  [gSpan] Found 1000 patterns so far (current code length: 8)
  [gSpan] Found 2000 patterns so far (current code length: 8)
  [gSpan] Found 3000 patterns so far (current code length: 8)
  [gSpan] Found 4000 patterns so far (current code length: 10)
  [gSpan] Found 5000 patterns so far (current code length: 5)
  [gSpan] Found 6000 patterns so far (current code length: 8)
  [gSpan] Found 7000 patterns so far (current code length: 8)
  [gSpan] Found 8000 patterns so far (current code length: 6)
  [gSpan] Found 9000 patterns so far (current code length: 8)
  [gSpan] Found 10000 patterns so far (current code length: 8)
  [gSpan] Found 11000 patterns so far (current code length: 9)
  [gSpan] Found 12000 patterns so far (current code length: 7)
  [gSpan] Found 13000 patterns so far (current code length: 7)
  [gSpan] Found 14000 patterns so far (current code length: 10)
  [gSpan] Found 15000 p

In [16]:
results = miner.get_frequent_subgraphs_as_nx()

print(f"\nTotal frequent subgraphs found: {len(results)}")
print(f"Vertex labels mapped: {miner.vlabel_map}")
print(f"Edge labels mapped: {miner.elabel_map}")


Total frequent subgraphs found: 29189
Vertex labels mapped: {'Ac': 0, 'Ag': 1, 'Al': 2, 'As': 3, 'Au': 4, 'B': 5, 'Bi': 6, 'Br': 7, 'C': 8, 'Cd': 9, 'Ce': 10, 'Cl': 11, 'Co': 12, 'Cr': 13, 'Cu': 14, 'Dy': 15, 'Er': 16, 'Eu': 17, 'F': 18, 'Fe': 19, 'Ga': 20, 'Gd': 21, 'Ge': 22, 'Hf': 23, 'Hg': 24, 'I': 25, 'In': 26, 'Ir': 27, 'K': 28, 'La': 29, 'Mg': 30, 'Mn': 31, 'Mo': 32, 'N': 33, 'Na': 34, 'Nb': 35, 'Nd': 36, 'Ni': 37, 'O': 38, 'Os': 39, 'P': 40, 'Pb': 41, 'Pd': 42, 'Pt': 43, 'Re': 44, 'Rh': 45, 'Ru': 46, 'S': 47, 'Sb': 48, 'Se': 49, 'Si': 50, 'Sm': 51, 'Sn': 52, 'Ta': 53, 'Te': 54, 'Th': 55, 'Ti': 56, 'Tl': 57, 'U': 58, 'V': 59, 'W': 60, 'Y': 61, 'Zn': 62, 'Zr': 63}
Edge labels mapped: {'DOUBLE': 0, 'SINGLE': 1, 'TRIPLE': 2}


In [17]:
size_dist = Counter(r['num_vertices'] for r in results)
print(f"Pattern size distribution:")
for s in sorted(size_dist):
    print(f"|V|={s}: {size_dist[s]} patterns")

Pattern size distribution:
|V|=2: 16 patterns
|V|=3: 51 patterns
|V|=4: 180 patterns
|V|=5: 575 patterns
|V|=6: 1471 patterns
|V|=7: 3087 patterns
|V|=8: 5247 patterns
|V|=9: 6721 patterns
|V|=10: 6095 patterns
|V|=11: 4010 patterns
|V|=12: 1736 patterns


In [18]:
print("\n--- Sample Frequent Subgraphs ---")
for i, r in enumerate(results[:15]):
    nodes = {n: d['feature'] for n, d in r['graph'].nodes(data=True)}
    edges = {(u, v): d['label'] for u, v, d in r['graph'].edges(data=True)}
    print(f"  #{i+1}: |V|={r['num_vertices']}, |E|={r['num_edges']}, "
          f"support={r['support']}")
    print(f"         nodes={nodes}")
    print(f"         edges={edges}")
    print(f"         DFS code: {r['dfs_code']}")


--- Sample Frequent Subgraphs ---
  #1: |V|=2, |E|=1, support=34314
         nodes={0: 'C', 1: 'C'}
         edges={(0, 1): 'DOUBLE'}
         DFS code: (0, 1, 8, 0, 8)
  #2: |V|=3, |E|=2, support=32996
         nodes={0: 'C', 1: 'C', 2: 'C'}
         edges={(0, 1): 'DOUBLE', (1, 2): 'SINGLE'}
         DFS code: (0, 1, 8, 0, 8) | (1, 2, 8, 1, 8)
  #3: |V|=4, |E|=3, support=29657
         nodes={0: 'C', 1: 'C', 2: 'C', 3: 'C'}
         edges={(0, 1): 'DOUBLE', (1, 2): 'SINGLE', (2, 3): 'DOUBLE'}
         DFS code: (0, 1, 8, 0, 8) | (1, 2, 8, 1, 8) | (2, 3, 8, 0, 8)
  #4: |V|=5, |E|=4, support=29424
         nodes={0: 'C', 1: 'C', 2: 'C', 3: 'C', 4: 'C'}
         edges={(0, 1): 'DOUBLE', (1, 2): 'SINGLE', (2, 3): 'DOUBLE', (3, 4): 'SINGLE'}
         DFS code: (0, 1, 8, 0, 8) | (1, 2, 8, 1, 8) | (2, 3, 8, 0, 8) | (3, 4, 8, 1, 8)
  #5: |V|=6, |E|=5, support=28469
         nodes={0: 'C', 1: 'C', 2: 'C', 3: 'C', 4: 'C', 5: 'C'}
         edges={(0, 1): 'DOUBLE', (1, 2): 'SINGLE', (2, 3): 'DO

In [19]:
# ===================================================================
# 5. Build binary indicator matrix (Def 2.3, CORK paper)
# ===================================================================
n_graphs = len(graphs)
n_features = len(results)

X = np.zeros((n_graphs, n_features), dtype=np.int8)
for feat_idx, r in enumerate(results):
    for gid in r['graph_ids']:
        X[gid, feat_idx] = 1

y_arr = np.array(y)

print(f"\nBinary indicator matrix: {X.shape}")
print(f"Sparsity: {1 - X.mean():.4f}")
print(f"Avg features per graph: {X.sum(axis=1).mean():.1f}")
print(f"Class distribution: {dict(zip(*np.unique(y_arr, return_counts=True)))}")


Binary indicator matrix: (37349, 29189)
Sparsity: 0.9149
Avg features per graph: 2484.4
Class distribution: {-1: 35556, 1: 1793}


In [20]:
type(X)

numpy.ndarray

In [21]:
len(X[0])

29189